Objective:- Goal is to predict potential project "Holds" based on supply chain disruptions due on external factors.
In the workflow, a State is used to flows between agents.
The Research Agent will use a search tool (like Tavily) to gather logistics news.
The Reasoning Agent will parse that data to predict potential "Holds."

pip install -U langgraph langchain-openai langchain-community tavily-python

In [ ]:
import operator
from typing import Annotated, List, TypedDict

from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.graph import StateGraph, END

# --- 1. Define State ---
# The state is the shared memory between your agents.
class AgentState(TypedDict):
    # 'operator.add' allows us to append new messages rather than overwriting
    messages: Annotated[List[BaseMessage], operator.add]
    research_data: str
    prediction: str

# --- 2. Initialize Tools & LLM ---
# Using GPT-4o for reasoning and Tavily for logistics news scraping
llm = ChatOpenAI(model="gpt-4o", temperature=0)
search_tool = TavilySearchResults(max_results=5)

# --- 3. Define the Nodes (Agents) ---

def research_agent(state: AgentState):
    """Scrapes global logistics news based on the input query."""
    last_message = state['messages'][-1].content
    
    # We execute the search tool
    # In a production app, you might use a specific logistics news API
    search_query = f"latest global supply chain disruptions logistics news {last_message}"
    results = search_tool.invoke({"query": search_query})
    
    # Format results for the next agent
    context = "\n".join([r['content'] for r in results])
    
    return {
        "research_data": context,
        "messages": [HumanMessage(content=f"Research complete. Found data on logistics disruptions.")]
    }

def reasoning_agent(state: AgentState):
    """Predicts project 'Holds' based on the researched logistics data."""
    context = state['research_data']
    
    prompt = f"""
    Based on the following logistics news, predict potential "Project Holds".
    Identify specific regions, routes, or materials that are at risk.
    
    News Context:
    {context}
    
    Format your response as:
    - RISK LEVEL: (Low/Medium/High)
    - PREDICTED HOLDS: (Specific disruptions)
    - REASONING: (Why these holds will occur)
    """
    
    prediction = llm.invoke(prompt)
    
    return {
        "prediction": prediction.content,
        "messages": [HumanMessage(content="Reasoning complete. Prediction generated.")]
    }

# --- 4. Build the Graph ---
workflow = StateGraph(AgentState)

# Add nodes
workflow.add_node("researcher", research_agent)
workflow.add_node("reasoner", reasoning_agent)

# Define edges (The flow: Start -> Research -> Reason -> End)
workflow.set_entry_point("researcher")
workflow.add_edge("researcher", "reasoner")
workflow.add_edge("reasoner", END)

# Compile the graph
app = workflow.compile()

# --- 5. Run the Workflow ---
if __name__ == "__main__":
    inputs = {
        "messages": [HumanMessage(content="Focus on semiconductor shipping from Taiwan to Europe.")]
    }
    
    final_state = app.invoke(inputs)
    
    print("\n--- FINAL LOGISTICS PREDICTION ---")
    print(final_state["prediction"])